In [0]:

import json# Mount

# Set up the configurations for mounting the GCS bucket
gcs_bucket_name = "gbmartecomdiviz2"
mount_point = "/mnt/mskltest_try5"
service_account_key = "/dbfs/FileStore/tables/mentorsko_1725955323137_10e09271ad89.json"

# Read the service account key file
with open(service_account_key, 'r') as key_file:
    service_account_info = json.load(key_file)

# Define the GCS service account credentials
config = {
    "fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
    "fs.gs.auth.service.account.enable": "true",
    "fs.gs.auth.service.account.email": service_account_info["client_email"],
    "fs.gs.auth.service.account.private.key.id": service_account_info["private_key_id"],
    "fs.gs.auth.service.account.private.key": service_account_info["private_key"],
    "fs.gs.project.id": service_account_info["project_id"]
}

if any(mount.mountPoint == mount_point for mount in dbutils.fs.mounts()):
    dbutils.fs.unmount(mount_point)

# Mount the GCS bucket
dbutils.fs.mount(
    source=f"gs://{gcs_bucket_name}",
    mount_point=mount_point,
    extra_configs=config
)

In [0]:
files = dbutils.fs.ls(f'{mount_point}/Ecom Dataset/')
display(files)

In [0]:
catalog_name = 'diviz_akkshat_databricks_npmentorskool_onmicrosoft_com'
schema_name = 'gcs_bronze'

files = dbutils.fs.ls(f'{mount_point}/Ecom Dataset/')

tables = []
for file in files:
    file_name = file.name[:-4]
        # Assuming file name without extension as table name
    tables.append(file_name)

print(tables)

In [0]:
# create the schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
 
# create the tables
for table in tables:
    spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog_name}.{schema_name}.{table}")
 
# Enable column mapping
spark.sql("SET spark.databricks.delta.columnMapping.mode = name")
spark.sql("SET spark.databricks.delta.schema.autoMerge.enabled = true")
 
# Now we want to load the data into tables by copy into the tables
for table in tables:
    spark.sql(f"""
        COPY INTO {catalog_name}.{schema_name}.{table}
        FROM '{mount_point}/Ecom Dataset/{table}.csv'
        FILEFORMAT = CSV
        FORMAT_OPTIONS (
            'header' = 'true',
            'inferSchema' = 'true',
            'timestampFormat' = 'dd-MM-yyyy HH.mm',
            'multiline' = 'true',
            'escape' = ')',
            'quote' = '"',
            'mergeSchema' = 'true'
        )
        COPY_OPTIONS ('mergeSchema' = 'true')
    """)

In [0]:
# List all tables in the specified schema
tables_df = spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}")
display(tables_df)

In [0]:
# List of tables to view
table_names = ['payments', 'orders', 'customers']  # Add the table names you want to view

# Loop through each table and display the records
for table_name in table_names:
    query = f"SELECT * FROM {catalog_name}.{schema_name}.{table_name}"
    table_df = spark.sql(query)
    display(table_df)